In [ ]:
from s03a_localisation import (
    compute_localised_non_linearity,
    show_localised_non_linearity, plot_cap
)
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(10,10))
plot_cap([1,0,0,1,]+[0,]*16+[1,]+[0,]*23+[1,]+[0,]*9, "Fp1, F4, TP8, O1", axes=plt.gca(), plane_distance=0.1)
plt.gca().axis("off")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import json, glob, os, re
import numpy as np
from matplotlib.patches import Patch, Rectangle
from matplotlib.colors import TwoSlopeNorm, LogNorm
from matplotlib.lines import Line2D
from matplotlib.axes import Axes
from scipy.stats import ttest_rel, pearsonr
from scipy.io import loadmat
import os, sys, configparser

settings_parser = configparser.ConfigParser()
settings_parser.read("localsettings.ini")
MAIN_DATA_FOLDER = settings_parser.get("global", "data_path")
MIENC_PATH = settings_parser.get("global", "mienc_path")

bands = {
    i: b
    for i, b in enumerate(
        [
            [1.0, 4.0],
            [4.0, 8.0],
            [8.0, 12.0],
            [12.0, 30.0],
            [30.0, 44.0],
        ],
        start=1,
    )
}

band_names = {
    i: b
    for i, b in enumerate(
        ["delta", "theta", "alpha", "beta", "gamma"],
        start=1,
    )
}

band_fancy_names = {
    i: b
    for i, b in enumerate(
        [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"],
        start=1,
    )
}



def fs_band(band):
    return int(bands[band][1] * 2 * 1.125 + 0.5)

nbins = [10, 13, 14, 20, 23]

# bands = ["delta", "theta", "alpha", "beta", "gamma"]
# band_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
sns.set_theme("paper", "ticks")
sns.set_palette("colorblind")

cache_dir = os.path.join(MAIN_DATA_FOLDER, "cache")
RESULTS_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Results")
FIGURES_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Figures")
if not os.path.isdir(FIGURES_FOLDER):
    os.mkdir(FIGURES_FOLDER)
if not os.path.isdir(os.path.join(RESULTS_FOLDER, "localised")):
    os.mkdir(os.path.join(RESULTS_FOLDER, "localised"))

sys.path.append(MIENC_PATH)
from mienc import Corrector

In [ ]:
pair_num = 1431#18915
subj_num = 150#50
elec_num = 54#195
results_file = os.path.join(
    RESULTS_FOLDER, "EEG_124s_band_{0}_bin{1}/subject{2:02}_{1}.npy"
)
subset_description = "EEG "
subset_identifiers = ["delta", "theta", "alpha", "beta", "gamma"]
subset_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
output_prefix = "EEG_54"
nbins = [10, 13, 14, 20, 23]
samples = []
for band, bins in zip(subset_identifiers, nbins):
    with open(
        os.path.join(os.path.dirname(results_file.format(band, bins, 0)), "shape.json")
    ) as fp:
        samples.append(json.load(fp)[0])

compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix,
    pair_num,
    subj_num,
    nbins,
    samples,
)

results_file = os.path.join(
    RESULTS_FOLDER, "EEG_124s_band_{0}_bin{1}/subject{2:02}_{1}_sha.npy"
)
compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix + "_sha",
    pair_num,
    subj_num,
    nbins,
    samples,
)

show_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    elec_num,
    FIGURES_FOLDER,
)